# Vega Trading — Strategy Research

This notebook demonstrates how to use the engine for strategy research and backtesting.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from engine.data.providers.synthetic import SyntheticDataProvider
from engine.strategies.pairs_trading import PairsTradingStrategy
from engine.strategies.mean_reversion import MeanReversionStrategy
from engine.backtester.engine import BacktestEngine

print('Engine loaded successfully')

## 1. Generate Synthetic Data

In [ ]:
provider = SyntheticDataProvider(seed=42, correlation=0.85)
data = provider.fetch_ohlcv(
    symbols=['SYN_A', 'SYN_B'],
    start_date='2022-01-01',
    end_date='2024-01-01',
)

print(f'Data shape: {data.shape}')
data['Close'].plot(figsize=(12, 4), title='Synthetic Correlated Prices')
plt.ylabel('Price ($)')
plt.show()

## 2. Pairs Trading Backtest

In [ ]:
strategy = PairsTradingStrategy(symbol_a='SYN_A', symbol_b='SYN_B')
engine = BacktestEngine(strategy=strategy, initial_capital=100000)
result = engine.run(data)

print('=== Performance Metrics ===')
for key, value in result['metrics'].items():
    if isinstance(value, float):
        print(f'{key}: {value:.4f}')
    else:
        print(f'{key}: {value}')

In [ ]:
equity = pd.DataFrame(result['equity_curve'])
equity['equity'].plot(figsize=(12, 4), title='Pairs Trading Equity Curve')
plt.ylabel('Portfolio Value ($)')
plt.axhline(y=100000, color='gray', linestyle='--', alpha=0.5, label='Initial Capital')
plt.legend()
plt.show()

## 3. Mean Reversion Backtest

In [ ]:
data_single = provider.fetch_ohlcv(
    symbols=['SYN_A'],
    start_date='2022-01-01',
    end_date='2024-01-01',
)

mr_strategy = MeanReversionStrategy(symbol='SYN_A')
mr_engine = BacktestEngine(strategy=mr_strategy, initial_capital=100000)
mr_result = mr_engine.run(data_single)

print('=== Mean Reversion Metrics ===')
for key, value in mr_result['metrics'].items():
    if isinstance(value, float):
        print(f'{key}: {value:.4f}')
    else:
        print(f'{key}: {value}')